# TP NLP — AG News (avec classe **Sports**)
**Dataset :** AG News (World/Sports/Business/Sci&Tech)  
**Page info :** https://paperswithcode.com/dataset/ag-news  
**Fichiers CSV :**  
- train : https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv  
- test : https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv

Ce notebook applique **exactement** les étapes vues en cours :  
1. **Nettoyage & prétraitement** (basique)  
2. **Représentation** (TF‑IDF pour ML, Tokenizer+Embedding pour DL)  
3. **Comparaison** de modèles **ML** (LogReg, Linear SVM) **vs** **DL** (CNN 1D simple)  
4. **Évaluation** multi‑classe (accuracy, macro‑precision/recall/F1, matrices de confusion, ROC‑AUC macro-ovo)

> Créé automatiquement le 2025-09-21 15:47 UTC. Exécuter cellule par cellule dans **Google Colab**.

In [1]:

import sys, os, re, io, random, zipfile, requests
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

DATA_DIR = Path("data_agnews"); DATA_DIR.mkdir(exist_ok=True)
OUT_DIR  = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("TensorFlow:", tf.__version__)


Python: 3.12.11
Pandas: 2.2.2
TensorFlow: 2.19.0


In [2]:
TRAIN_URL = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
TEST_URL  = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv"

train_path = DATA_DIR / "ag_news_train.csv"
test_path  = DATA_DIR / "ag_news_test.csv"

if not train_path.exists():
    train_path.write_bytes(requests.get(TRAIN_URL, timeout=60).content)
if not test_path.exists():
    test_path.write_bytes(requests.get(TEST_URL, timeout=60).content)

# Les CSV sont au format: label, title, description
columns = ['label','title','text']
train_df = pd.read_csv(train_path, header=None, names=columns)
test_df  = pd.read_csv(test_path,  header=None, names=columns)

print(train_df.shape, test_df.shape)
train_df.head()


(120000, 3) (7600, 3)


,label,title,text
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [3]:

# Mapping des labels (1..4) -> noms
label_map = {1:"World", 2:"Sports", 3:"Business", 4:"Sci/Tech"}
train_df['label_name'] = train_df['label'].map(label_map)
test_df['label_name']  = test_df['label'].map(label_map)

# Fusionner title + text pour un signal unifié
train_df['text_all'] = (train_df['title'].astype(str) + " " + train_df['text'].astype(str)).str.strip()
test_df['text_all']  = (test_df['title'].astype(str) + " " + test_df['text'].astype(str)).str.strip()

print(train_df['label_name'].value_counts())
train_df[['label','label_name','title']].head(8)


label_name
Business    30000
Sci/Tech    30000
Sports      30000
World       30000
Name: count, dtype: int64


,label,label_name,title
0,3,Business,Wall St. Bears Claw Back Into the Black (Reuters)
1,3,Business,Carlyle Looks Toward Commercial Aerospace (Reu...
2,3,Business,Oil and Economy Cloud Stocks' Outlook (Reuters)
3,3,Business,Iraq Halts Oil Exports from Main Southern Pipe...
4,3,Business,"Oil prices soar to all-time record, posing new..."
5,3,Business,"Stocks End Up, But Near Year Lows (Reuters)"
6,3,Business,Money Funds Fell in Latest Week (AP)
7,3,Business,Fed minutes show dissent over inflation (USATO...


## 1) Nettoyage & prétraitement (basique — comme en cours)

- minuscule
- normalisation espaces
- suppression URLs/emails/hashtags/mentions
- retrait ponctuation/chiffres
- petite liste de stopwords (simple)

In [4]:

STOPWORDS = {
    "a","an","the","and","or","if","in","on","at","of","for","to","from","by","with","as",
    "is","are","was","were","be","been","being","do","does","did","doing",
    "this","that","these","those","it","its","i","you","he","she","we","they","me","him","her","us","them",
    "my","your","his","our","their"
}

def basic_clean(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\\S+|www\\.\\S+", " URL ", text)
    text = re.sub(r"\\S+@\\S+", " EMAIL ", text)
    text = re.sub(r"[@#]\\w+", " TAG ", text)
    text = re.sub(r"[^a-z\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    tokens = [w for w in text.split() if w not in STOPWORDS]
    return " ".join(tokens)

train_df['clean'] = train_df['text_all'].astype(str).apply(basic_clean)
test_df['clean']  = test_df['text_all'].astype(str).apply(basic_clean)

train_df[['text_all','clean','label_name']].head(6)


,text_all,clean,label_name
0,Wall St. Bears Claw Back Into the Black (Reute...,wall st bears claw back into black reuters reu...,Business
1,Carlyle Looks Toward Commercial Aerospace (Reu...,carlyle looks toward commercial aerospace reut...,Business
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,oil economy cloud stocks outlook reuters reute...,Business
3,Iraq Halts Oil Exports from Main Southern Pipe...,iraq halts oil exports main southern pipeline ...,Business
4,"Oil prices soar to all-time record, posing new...",oil prices soar all time record posing new men...,Business
5,"Stocks End Up, But Near Year Lows (Reuters) Re...",stocks end up but near year lows reuters reute...,Business


In [5]:

# Comme en cours : on conserve le test officiel, et on split le TRAIN en train/val
X = train_df['clean'].values
y = train_df['label'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=SEED, stratify=y)
X_test,  y_test  = test_df['clean'].values, test_df['label'].values

print(len(X_train), len(X_val), len(X_test))
target_names = ["World","Sports","Business","Sci/Tech"]


108000 12000 7600


## 2) Représentation (TF‑IDF) & 3) Modèles ML

Modèles simples vus en cours :  
- **Logistic Regression (multi‑classe)**  
- **Linear SVM**  

Sélection par **macro‑F1** sur validation, puis ré‑apprentissage sur **train+val** et test final.

In [6]:

from sklearn.metrics import log_loss

def macro_scores(y_true, y_pred, probas=None):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    out = {"accuracy":acc, "precision_macro":prec, "recall_macro":rec, "f1_macro":f1}
    if probas is not None and len(np.unique(y_true)) > 2:
        try:
            # besoin de scores par classe pour AUC macro-ovo; ici on approx avec one-vs-rest
            from sklearn.preprocessing import label_binarize
            classes = np.unique(y_true)
            y_bin = label_binarize(y_true, classes=classes)
            auc = roc_auc_score(y_bin, probas, average='macro', multi_class='ovr')
            out["roc_auc_macro_ovr"] = auc
        except Exception:
            pass
    return out

results = {}

# TF-IDF commun
tfidf = TfidfVectorizer(max_features=40000, ngram_range=(1,2))

# Logistic Regression
pipe_lr = Pipeline([("tfidf", tfidf), ("clf", LogisticRegression(max_iter=400, multi_class="ovr"))])
gs_lr = GridSearchCV(pipe_lr, {"clf__C":[0.5,1.0,2.0]}, cv=3, n_jobs=-1, scoring="f1_macro", verbose=0)
gs_lr.fit(X_train, y_train)
pred_val_lr = gs_lr.predict(X_val)
# Probabilités (pour AUC)
try:
    proba_val_lr = gs_lr.predict_proba(X_val)
except Exception:
    proba_val_lr = None
results["LogReg_val"] = macro_scores(y_val, pred_val_lr, proba_val_lr)

# Linear SVM
pipe_svm = Pipeline([("tfidf", tfidf), ("clf", LinearSVC())])
gs_svm = GridSearchCV(pipe_svm, {"clf__C":[0.5,1.0,2.0]}, cv=3, n_jobs=-1, scoring="f1_macro", verbose=0)
gs_svm.fit(X_train, y_train)
pred_val_svm = gs_svm.predict(X_val)
# Scores pour AUC : decision_function -> min-max scale
dfc = gs_svm.decision_function(X_val)
dfc_scaled = (dfc - dfc.min())/(dfc.max()-dfc.min()+1e-9)
results["LinearSVM_val"] = macro_scores(y_val, pred_val_svm, dfc_scaled)

results


{'LogReg_val': {'accuracy': 0.9255,
  'precision_macro': 0.9253279693443209,
  'recall_macro': 0.9255,
  'f1_macro': 0.9252966610369859,
  'roc_auc_macro_ovr': np.float64(0.9863582592592592)},
 'LinearSVM_val': {'accuracy': 0.9266666666666666,
  'precision_macro': 0.9265300918507885,
  'recall_macro': 0.9266666666666666,
  'f1_macro': 0.9264825792807222,
  'roc_auc_macro_ovr': np.float64(0.9855632314814814)}}

In [ ]:

# Choix du meilleur modèle sur validation (macro F1)
best_name = max(results.items(), key=lambda kv: kv[1]["f1_macro"])[0]
best_estimator = gs_lr if "LogReg" in best_name else gs_svm
print("Meilleur modèle (val) :", best_name)

# Fit sur train+val puis test
best_estimator.fit(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]))
y_pred_test = best_estimator.predict(X_test)

# Probabilités/scores
if hasattr(best_estimator, "predict_proba"):
    y_score_test = best_estimator.predict_proba(X_test)
elif hasattr(best_estimator, "decision_function"):
    sc = best_estimator.decision_function(X_test)
    y_score_test = (sc - sc.min())/(sc.max()-sc.min()+1e-9)
else:
    y_score_test = None

ml_metrics_test = macro_scores(y_test, y_pred_test, y_score_test)
print("Scores test (ML) :", ml_metrics_test)

print("\nClassification report (ML):\n",
      classification_report(y_test, y_pred_test, target_names=target_names, digits=4))

cm_ml = confusion_matrix(y_test, y_pred_test, labels=[1,2,3,4])
cm_ml


## 2) Représentation (DL) & 3) Modèle DL (CNN 1D simple)

Étapes basiques :  
- **Tokenizer** (20k mots) + **padding**  
- **Embedding(128)** + **Conv1D(64)** + **GlobalMaxPooling** + **Dropout** + **Dense softmax(4)**  
- **EarlyStopping** sur `val_loss`  

In [ ]:

VOCAB = 20000
tokenizer = Tokenizer(num_words=VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

def to_seq(texts): return tokenizer.texts_to_sequences(texts)

seq_tr = to_seq(X_train)
seq_va = to_seq(X_val)
seq_te = to_seq(X_test)

lengths = list(map(len, seq_tr))
MAX_LEN = int(np.percentile(lengths, 95))
MAX_LEN = max(30, min(MAX_LEN, 250))
print("MAX_LEN:", MAX_LEN)

Xtr = pad_sequences(seq_tr, maxlen=MAX_LEN, padding='post', truncating='post')
Xva = pad_sequences(seq_va, maxlen=MAX_LEN, padding='post', truncating='post')
Xte = pad_sequences(seq_te, maxlen=MAX_LEN, padding='post', truncating='post')

# Labels 1..4 -> 0..3 one-hot
num_classes = 4
ytr = tf.keras.utils.to_categorical(y_train-1, num_classes)
yva = tf.keras.utils.to_categorical(y_val-1,   num_classes)
yte = tf.keras.utils.to_categorical(y_test-1,  num_classes)

model = Sequential([
    Embedding(input_dim=VOCAB+1, output_dim=128, input_length=MAX_LEN),
    Conv1D(64, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

es = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
hist = model.fit(Xtr, ytr, validation_data=(Xva, yva),
                 epochs=12, batch_size=128, callbacks=[es], verbose=1)


In [ ]:

proba_dl = model.predict(Xte, verbose=0)
pred_dl  = proba_dl.argmax(axis=1) + 1  # back to 1..4
dl_metrics = macro_scores(y_test, pred_dl, proba_dl)
print("Scores test (DL):", dl_metrics)

print("\nClassification report (DL):\n",
      classification_report(y_test, pred_dl, target_names=target_names, digits=4))

cm_dl = confusion_matrix(y_test, pred_dl, labels=[1,2,3,4])
cm_dl


In [ ]:

# Courbes d'entraînement
plt.figure(figsize=(5,4))
plt.plot(hist.history['loss'], label='train')
plt.plot(hist.history['val_loss'], label='val')
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("CNN 1D — Loss"); plt.legend(); plt.grid(True); plt.show()

plt.figure(figsize=(5,4))
plt.plot(hist.history['accuracy'], label='train')
plt.plot(hist.history['val_accuracy'], label='val')
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("CNN 1D — Accuracy"); plt.legend(); plt.grid(True); plt.show()


In [ ]:

# Comparatif final (test)
import pandas as pd
summary = pd.DataFrame([
    {"model":"ML (best)", **ml_metrics_test},
    {"model":"DL (CNN1D)", **dl_metrics}
])
summary


## Commentaires (dans le notebook, pas de rapport séparé)

- Le **prétraitement** est volontairement **basique** (comme en cours) : pas de lemmatisation ni techniques avancées.  
- **TF‑IDF + Linear SVM** est une **baseline forte** sur AG News ; **Logistic Regression** est proche.  
- Le **DL** avec une **CNN 1D simple** atteint des performances comparables voire meilleures si on augmente un peu l’entraînement, mais on reste **sobre** (pas de Transformers).  
- Métrique choisie : **macro‑F1** (équilibre les 4 classes).  
- Pour aller plus loin (optionnel) : stopwords plus riches, n-grams 1–3, lemmatisation, pré‑embeddings (GloVe), ou BERT — **non requis ici**.